# Genomic structural variation across clownfish genomes

Scripts and pipeline associated with **Marcionetti et al.**, *"Genomic structural
variation across clownfish genomes"*.

This single notebook documents the full analysis, from raw PacBio HiFi data to the
structural variant (SV) catalogue and downstream analyses. Every numbered section
maps to a subsection of the manuscript *Methods*.

## How to read this notebook

* Code cells are the **exact commands / script invocations** used, in order. They
  are documentation of the workflow, not a script to run top-to-bottom: most steps
  are SLURM array jobs on large genomic files that are not part of this repository.
* All helper scripts live under `scripts/<section>/`; small input tables live under
  `metadata/`. Large inputs/outputs (assemblies, BAMs, VCFs, the SV-catalogue HTML,
  `.xlsx` tables, PDFs) are archived separately - see `README.md`.
* Third-party tools invoked by name (`plotsr`, `pafCoordsDotPlotly.R`, AGORA's
  `misc.compareGenomes.py`, `AMAS.py`, …) are listed with their sources in
  `README.md`; 

## Study species

16 clownfish genomes newly assembled here, complemented with publicly available
assemblies and outgroups:

| ID | Species | Platform | Role |
|----|---------|----------|------|
| AKA | *Amphiprion akallopisos* | HiFi Sequel II | assembled |
| AKY | *A. akindynos* | HiFi Sequel II | assembled |
| ALL | *A. allardi* | HiFi Revio | assembled |
| BIA | *A. biaculeatus* | HiFi Revio | assembled |
| CRP | *A. chrysopterus* | HiFi Revio | assembled |
| EPH | *A. ephippium* | HiFi Sequel II | assembled |
| FRE | *A. frenatus* | HiFi Revio | assembled |
| LAT | *A. latifasciatus* | HiFi Sequel II | assembled |
| LAZ | *A. latezonatus* | HiFi Revio | assembled |
| MCC | *A. mccullochi* | HiFi Revio | assembled |
| OMA | *A. omanensis* | HiFi Sequel II | assembled |
| POL | *A. polymnus* | HiFi Sequel II | assembled |
| PRC | *A. percula* | HiFi Revio | assembled |
| PRD | *A. perideraion* | HiFi Sequel II | assembled |
| SAN | *A. sandaracinos* | HiFi Revio | assembled |
| SEB | *A. sebae* | HiFi Revio | assembled |
| BIARef | *A. biaculeatus* | - | public ref (GCA_053813585.1) |
| CLA | *A. clarkii* | - | public ref (GCA_027123335.1) |
| OCE | *A. ocellaris* | - | public ref (GCA_022539595.1) |
| PRCRef | *A. percula* | - | public ref (GCA_003047355.2) |
| DTR | *Dascyllus trimaculatus* | - | outgroup (GCA_024666655.1) |
| ACH | *Acanthochromis polyacanthus* | - | outgroup (GCA_021347895.1) |

Species lists used throughout: `metadata/Species.txt` (16 assembled),
`metadata/Species_with_outgroups.txt`, `metadata/Species_OMA.txt`.

---
## 1. PacBio data and preprocessing

### 1.1 Read QC
HiFi read quality assessed with **FastQC** (v0.12.1), summarised with **MultiQC** (v1.25.1).

In [ ]:
sbatch scripts/1_PacBio_Data_and_Preprocessing/Run_FastQC.sh <workdir> <species_list.txt>

multiqc --outdir <out_dir> <fastqc_dir>

### 1.2 Genome characteristics
k-mer counting with **Jellyfish** (v2.2.7, k=21) and **GenomeScope** (v1.0).

In [ ]:
bash scripts/1_PacBio_Data_and_Preprocessing/Submit_Jellyfish.sh <species_list.txt>

bash scripts/1_PacBio_Data_and_Preprocessing/Submit_Genoscope.sh <species_list.txt> <jellyfish_dir>

---
## 2. Genome assembly

### 2.1 Assembler evaluation
For four species (AKA, PRD, EPH, POL) four assemblers were compared: **IPA** (v1.8.0),
**Flye** (v2.9.1), **HiCanu** (v2.2) and **Hifiasm** (v0.19.5). Quality assessed with
`assembly-stats` (v1.0.1), **BUSCO** (v5.4.7, `actinopterygii_odb10`) and by mapping
HiFi reads back with **minimap2** (v2.12) + **samtools** (v1.19.2). Haplotypic
duplication removed with **purge_dups** (v1.2.5). Hifiasm performed best and was
selected (Supplementary Table S15).

In [ ]:
sbatch scripts/2_Genome_Assembly/Run_ipa.sh <workdir> <reads.fastq>

sbatch scripts/2_Genome_Assembly/Run_flye.sh <workdir> <reads.fastq>

sbatch scripts/2_Genome_Assembly/Run_canu.sh <workdir> <reads.fastq> <genome_size> <out_prefix>

sbatch scripts/2_Genome_Assembly/Run_hifiasm.sh <workdir> <reads.fastq> <out_prefix>

# Evaluation
sbatch scripts/2_Genome_Assembly/Run_AssemblyStats.SoftwareEvaluation.sh <workdir> <species>

sbatch scripts/2_Genome_Assembly/Run_Busco.sh <out_dir> <assembly.fa> <out_prefix>

sbatch scripts/2_Genome_Assembly/Run_minimap2_forStats.sh <out_dir> <assembly.fa> <index.mmi> <reads.fastq> <out_prefix>

# purge_dups (3 steps) on Hifiasm + HiCanu assemblies
sbatch scripts/2_Genome_Assembly/Step1a_PurgeDup.sh <workdir> <new_dir> <assembly.fa> <reads.fastq> <out_prefix>   # PAF

sbatch scripts/2_Genome_Assembly/Step1b_PurgeDup.sh <workdir> <assembly.fa> <split_assembly.fa> <out_prefix>       # self-self alignment

sbatch scripts/2_Genome_Assembly/Step2_PurgeDup.sh  <workdir> <out_prefix> <assembly.fa>                          # purge + split

### 2.2 Assembly of all species - Hifiasm + purge_dups

In [ ]:
sbatch scripts/2_Genome_Assembly/Run_hifiasm.sh <workdir> <reads.fastq> <out_prefix>

# purge_dups for all species (Step1a / Step1b / Step2 as above)
sbatch scripts/2_Genome_Assembly/Step1a_PurgeDup.sh <workdir> <new_dir> <assembly.fa> <reads.fastq> <out_prefix>

sbatch scripts/2_Genome_Assembly/Step1b_PurgeDup.sh <workdir> <assembly.fa> <split_assembly.fa> <out_prefix>

sbatch scripts/2_Genome_Assembly/Step2_PurgeDup.sh  <workdir> <out_prefix> <assembly.fa>

### 2.3 Sequential MUMmer + mitochondrial-sequence removal
Duplicated scaffolds (≥80 % length matching another scaffold at >80 % identity) and
mitochondrial contigs identified with **MUMmer4** (v4.0.0; `nucmer`, `delta-filter`,
`show-coords`) and removed with custom scripts.

In [ ]:
sbatch scripts/2_Genome_Assembly/Run_Mummer_Sequential.sh <assembly.fa> <tmp_dir> <out_prefix> <identity_frac> <coverage_pct> scripts/2_Genome_Assembly/Sequential_mummer.py

# find + remove mitochondrial sequence
sbatch scripts/2_Genome_Assembly/Run_Mummer.sh <workdir> <reference_mito.fa> <query.fa> <out_prefix>

python scripts/2_Genome_Assembly/Extract_Sequences_from_fasta.py <in.fasta> <mito_id> <mito.fasta> <remaining.fasta>

### 2.4 Scaffold length & whole-genome alignment to *A. clarkii*
Obtain scaffold length and align assemblies to the *A. clarkii* reference genome (GCA_027123335.1). Whole genome alignmnets visualized with dotPlotly (https://github.com/tpoorten/dotPlotly), scripts `pafCoordsDotPlotly.R` and `mummerCoordsDotPlotly.R`

In [ ]:
python scripts/2_Genome_Assembly/Get_Contig_length.py <in.fasta> <out.tsv>

sbatch scripts/2_Genome_Assembly/Run_Mummer.sh <workdir> <clarkii_reference.fa> <assembly.fa> <out_prefix>

### 2.5 Identify & split potential misassemblies (based on *A. clarkii* alignments)

In [ ]:
python scripts/2_Genome_Assembly/Investigate_MisAssemblies.py <coords.tsv> <scaffold_lengths.txt> <pct_thresh> <cov_thresh> <summary_out.txt> > <split_candidates.txt>

python scripts/2_Genome_Assembly/Find_splitting_points.py <coords.tsv> <split_candidates.txt> <out_prefix>

# manual curation of split coordinates, then:
python scripts/2_Genome_Assembly/Split_misassemblies.py <in.fasta> <out.fasta> <split_coords.txt>

### 2.6 Reference-guided scaffolding
Scaffolds grouped by *A. clarkii*  chromosome, then scaffolded per cluster with **SAMBA** (MaSuRCA v4.1.0) + **ntLink** (v1.3.9).

In [ ]:
python scripts/2_Genome_Assembly/Investigate_WGAlignments.py <scaffold_list.txt> <coords.tsv> <out_prefix> <min_pct_aligned>

python scripts/2_Genome_Assembly/Prepare_chromosomes_for_scaffolding.py <total_scaffolds.txt> <best_alignments.txt> <all_alignments.txt> <scaffolding_groups_out.txt>

sbatch scripts/2_Genome_Assembly/Separate_Assembly_By_Chrom.sh <workdir> <assembly.fa> <species>

sbatch --array 1-N scripts/2_Genome_Assembly/AssemblyCuration.byChromosomes.sh <workdir> <species> <threads> <reads.fastq>

python scripts/2_Genome_Assembly/Reformat_scaffolded_Assemblies.py <species> <out.fasta> <suffix>

### 2.7 Pseudo-PE scaffolding
Paired-end-like reads extracted from HiFi reads at several insert sizes (5, 7, 9, 11 and 15 kb),
mapped with **BWA** (v0.7.17), and used for scaffolding with **P_RNA_scaffolder**
(5 rounds). Finally, scaffold shorter than `min_length_bp` = 200kb were removed.

In [ ]:
# extract pseudo-PE reads (per insert size)
sbatch scripts/2_Genome_Assembly/Run_ExtractPER_General.sh <reads_dir> <out_prefix> <min_hifi_len> <insert_size> <start_offset> <suffix> <scripts_dir>

In [ ]:
# map + scaffold (repeat rounds 1..5)
sbatch scripts/2_Genome_Assembly/Run_BWAIndex_forScaff.sh <workdir> <index_name> <assembly.fa>
for pe in pe5000 pe7000 pe9000 pe11000 pe15000; do
    sbatch scripts/2_Genome_Assembly/Run_BWA_forScaff.sh <workdir> <index_name> $pe <out.sam>
    sbatch scripts/2_Genome_Assembly/Run_pRNAScaff.sh  <workdir> <sam> <assembly.fa> $pe <out_prefix>
 done
python scripts/2_Genome_Assembly/Extract_Scaffolded_Info_pRNAScaff.py <species_list.txt> <chrom_map.txt> <scaffold_dir> <out_dir> <step_label>
python scripts/2_Genome_Assembly/Combine_reliable_connections.ChromInfo.py <dir> <species> <out> <min_cov=5> <max_cov=25> <chrom_map.txt>
bash   scripts/2_Genome_Assembly/p_RNA_Scaff_step2.sh <connections_file> <assembly.fa>
python scripts/2_Genome_Assembly/Rename_ScaffoldedSequences.py <in.fasta> <out.fasta> <round_prefix>

In [ ]:
# After the 5 rounds of scaffolding, remove scaffold shorter than min_length_bp
python scripts/2_Genome_Assembly/Remove_Short_Scaffolds.py <in.fasta> <out.fasta> <min_length_bp>

### 2.8 Scaffold renaming
Final scaffolds renamed by homology to *A. clarkii* chromosomes: `scf01.1 … scf24.1`
(`.2`, `.3` for additional scaffolds of the same chromosome; `scf25+` for unplaced).

In [ ]:
python scripts/2_Genome_Assembly/Get_Chromosomes_Scaffolds_Association.py <coords.tsv> <out.txt>
python scripts/2_Genome_Assembly/Rename_Scaffolds_withChromInfo.py -f <in.fasta> -o <out.fasta> -c <chrom_association.txt>

---
## 3. Genome quality and completeness

### 3.1 Scaffold length, assembly-stats, BUSCO
`assembly-stats` (v1.0.1) and **BUSCO** (v5.4.7, `actinopterygii_odb10`) at every assembly step (Supplementary Table S2).

In [ ]:
python scripts/2_Genome_Assembly/Get_Contig_length.py <in.fasta> <out.tsv>

assembly-stats -t <assemblies...> > <out.txt>

sbatch scripts/3_Genome_Quality/Run_Busco.sh <out_dir> <assembly.fa> <out_prefix>

### 3.2 Inspector
**Inspector** (v1.3.1) run at every assembly step; corrected assemblies re-evaluated to check convergence.

In [ ]:
sbatch scripts/3_Genome_Quality/Run_Inspector.sh <workdir> <assembly.fa> <reads.fastq> <out_folder>

python scripts/3_Genome_Quality/Get_Summary_Inspector_AllSpecies.py <inspector_dir> <out.txt>

python scripts/3_Genome_Quality/Get_Summary_Inspector_AllSpecies_Corrected.py <inspector_dir> <out.txt>

### 3.3 Individual heterozygosity & residual-error check (HiFi remapping)
HiFi reads mapped back to the final assemblies with **minimap2** (v2.28, `-x map-hifi`);
SNPs called with **Clair3** (v2.0.1, `--platform=hifi`, `hifi_sequel2` / `hifi_revio`
model per platform), SVs with **Sniffles2** (v2.5). Depth / SNP / SV statistics with
**VCFtools** (v0.1.16). Individual heterozygosity computed in non-overlapping 100-kb
windows; windows with <50 SNPs excluded.

In [ ]:
# 1. map HiFi reads to each assembly 
sbatch scripts/3_Genome_Quality/Run_HiFi_Mapping.sh <species_list.txt> <workdir>
sbatch scripts/3_Genome_Quality/Run_sam2bam.sh <species_list.txt> <bam_dir>

# 2. SVs (Sniffles2) 
sbatch scripts/3_Genome_Quality/Run_sniffle2.sh <species_list.txt> <workdir>
python scripts/3_Genome_Quality/summarize_self_mapping_SVs.py <workdir>
# plots: Plot_sniffle_out.R 

# 3. SNPs (Clair3) 
sbatch --array scripts/3_Genome_Quality/Run_clair3.sh <species_list.txt> <in_dir> <out_dir> <hifi_sequel2|hifi_revio>
bash scripts/3_Genome_Quality/SNPs_By_Reference_Stats.sh
# stats/plots: SNPs_By_Reference_Stats.R 

# 4. heterozygosity in 100-kb windows (>=50 SNPs)
sbatch scripts/3_Genome_Quality/Run_het_windows_selfmap.sh <in_dir> <species_list.txt>
# plots: plot_windows_het.R  /  plot_mapping_stats.R

---
## 4. Genome annotation
Annotation performed on the Inspector-corrected assemblies. Annotation also performed for the *Dascyllus trimaculatus* reference genome (GCA\_024666655.1). 

### 4.1 Repeat modelling & masking
Repeats modelled with **EDTA** (v2.1.3) and **RepeatModeler2** (v2.0.1, `-LTRStruct`),
libraries merged with the **DFAM Vertebrata** library (release 3.3) and the
clownfish-specific library from Marcionetti & Salamin (2023). Soft- and hard-masking
with **RepeatMasker** (v4.0.7); repeat landscapes from RepeatMasker utilities.

In [ ]:
sbatch scripts/4_Genome_Annotation/Run_EDTA.sh <out_dir> <assembly.fa>

sbatch scripts/4_Genome_Annotation/Run_RepeatModeler.sh <workdir> <assembly.fa> <db_name>

# Concatenate the libraries 
cat <edta_lib.fa> <repeatmodeler_lib.fa> <dfam_lib.fa> <clownfish_lib.fa> > <combined_lib.fa>

# Soft- and hard-masking with RepeatMasker
sbatch scripts/4_Genome_Annotation/Run_RepeatMasker.sh <workdir> <assembly.fa> <combined_lib.fa> <out_prefix> SM
sbatch scripts/4_Genome_Annotation/Run_RepeatMasker.sh <workdir> <assembly.fa> <combined_lib.fa> <out_prefix_hm> HM

### 4.2 Structural annotation - BRAKER3
**BRAKER** (v3.0.3) on the soft-masked assemblies with OrthoDB Eukaryota v11 protein evidence.

In [ ]:
sbatch scripts/4_Genome_Annotation/Run_Braker.sh <workdir> <assembly.softmasked.fa> <species_id>

bash scripts/4_Genome_Annotation/Rename_braker_output.sh <species_list.txt>

python scripts/4_Genome_Annotation/Braker_Annotation_Stats.py <braker_dir> <in_prefix> <out_prefix> > <stats>

### 4.3 Evaluation & filtering of gene models
Homology to **Swiss-Prot** (release 2024_1) with **DIAMOND** (v2.0.15) and domains
with **InterProScan** (v5.66-98.0; Pfam v36, PANTHER v18, AntiFam). Genes with no
hit in either database removed.

In [ ]:
diamond makedb --in <swissprot.fasta> --db <swissprot.dmnd>
sbatch scripts/4_Genome_Annotation/Run_Diamond.sh <workdir> <proteins.fa> <swissprot.dmnd> <out_prefix>

# InterProScan:
sbatch scripts/4_Genome_Annotation/Run_InterProScan.sh <workdir> <species> <proteins.fa> Pfam,AntiFam,PANTHER

# Evaluation Both DB
python scripts/4_Genome_Annotation/Databases_Matches_Evaluation.py <proteins.fa> <swissprot_hits> <interpro_hits> <out_prefix>

# Filter the annotation
python scripts/4_Genome_Annotation/Filter_annotation.py <out_dir> <braker_prefix> <out_prefix> <genes_to_keep.txt>
python scripts/4_Genome_Annotation/Filter_annotation_StatsFile.py <braker_summary> <genes_to_keep.txt> <out_summary>

### 4.4 Functional annotation
**DIAMOND** blastp against NCBI **nr** (2024-02-07); GO from NCBI `gene2go` / `gene2accession` / `gene_info` (2024-04-02).

In [ ]:
sbatch scripts/4_Genome_Annotation/Run_Diamond.sh <workdir> <proteins.fa> <nr.dmnd> <out_prefix>

python scripts/4_Genome_Annotation/Get_Best_AnnotatedSequences.py <diamond_hits> <out.blastp> 

python scripts/4_Genome_Annotation/Reduce_ncbi_genefiles.py <best_blast.blastp> <ncbi_db_dir> <out_prefix>

python scripts/4_Genome_Annotation/Get_final_functional_annotation.py <proteins.fa> <longest_iso.fa> <best_blast.blastp> <ncbi_prefix> <out.txt>

### 4.5 Annotation completeness - BUSCO & OMArk

In [ ]:
sbatch scripts/4_Genome_Annotation/Run_Busco_Annotation.sh <workdir> <proteins.fa> <out_prefix>

python scripts/4_Genome_Annotation/Get_splice_files.py <proteins.fa> <out.splice>

sbatch scripts/4_Genome_Annotation/Run_omamer.sh <proteins.fa> <out.omamer> <db>

sbatch scripts/4_Genome_Annotation/Run_omark.sh <workdir> <species> <db>

python scripts/4_Genome_Annotation/plot_all_results.OMArk.py -i <omark_dir> -m <mapping.txt> -o <out.png> -t

### 4.6  Genome-content plots (Circos)
**Circos** (v0.69-8); GC in 10-kb windows (`Calculate_GC_content.py`), gene / repeat density with **deepStats**.

In [ ]:
samtools faidx <assembly.fa>  # index the assembly

gtf2bed < <annotation.gtf> > <annotation.bed>  # convert annotation to bed

python scripts/4_Genome_Annotation/Calculate_GC_content.py <assembly.fa> <out.bedGraph> <window_bp>

# deepStats for gene and repeat density
dsComputeBEDDensity --input <features.bed> -c <assembly.fa.fai> -w <window_bp> -o <out_prefix>

# Plot circos
circos -conf <circos.conf>

---
## 5. Phylogenomics, OMA & ancestral scaffolds

### 5.1 Orthology - OMA
**OMA standalone** run on the annotated proteomes of the assembled species
plus DTR, ACH, and OMA-database genomes for *A. ocellaris*, *A. percula*,
*Oreochromis aureus* and *O. niloticus* (outgroups). Stop codons stripped; one
isoform per gene via `.splice` files (Section 4.5). HOGs summarised with **pyHAM**.

In [ ]:
for f in *.fa; do sed -i 's/\*//g' "$f"; done # strip stop codons

./bin/oma -c  # build DB

sbatch scripts/5_Phylogenomics_OMA_AGORA/Run_AllvsAll_OMA.sh

sbatch scripts/5_Phylogenomics_OMA_AGORA/Run_OMA_part3.sh

python scripts/5_Phylogenomics_OMA_AGORA/Analyse_OMA_Output.py

# single-copy orthologous genes (1:1 OGs) shared among all clownfish species and at least one damselfish species
python scripts/5_Phylogenomics_OMA_AGORA/Get_1to1_OG.py <in_file=HOGgenes_Damselfish.txt> <out_file>

### 5.2 Species tree from single-copy HOGs
1-to-1 orthologues across clownfishes + damselfish aligned with **MAFFT** (v7.505),
trimmed with **trimAl**, concatenated (**AMAS**), and a partitioned tree inferred
with **IQ-TREE** (v2.2.2.7, `-m GTR+G+I -p partitions -alrt 1000 -bb 1000`).
Damselfish (*D. trimaculatus*, *A. polyacanthus*) used as outgroups, then pruned.
Visualised with **iTOL** (v7.5.1). Result: `metadata/PacBioSpecies.HOG_Tree.treefile`.

In [ ]:
python scripts/5_Phylogenomics_OMA_AGORA/Get_HOGposition_CLAChrom.py <gene_positions.txt> <hogs.txt> <out.txt>

sbatch scripts/5_Phylogenomics_OMA_AGORA/Run_mafft.sh <workdir> <og_list.txt> <in_dir> <out_dir> <trimal_out_dir>

sbatch scripts/5_Phylogenomics_OMA_AGORA/Run_IQTree_HOGs.sh   # concat + GTR+G+I partitioned, outgroup DTR

### 5.3 Ancestral scaffolds - AGORA
**AGORA** (v3.1, *agora-generic*) with the 40,823 inferred HOGs as gene families.
Extant vs reconstructed-ancestor orthology compared with AGORA's
`src/misc.compareGenomes.py` (`printOrthologuesList`, `printOrthologousChrom`;
scaffolds/chromosomes >200 genes). Karyotype plots via `misc.compareGenomes.py`
`-mode=drawKaryotype` and **syntenyPlotteR** (v1.0).

In [ ]:
# Formatting for AGORA
python scripts/5_Phylogenomics_OMA_AGORA/Format_HOGs_For_AGORA.py     

# Run AGORA
sbatch scripts/5_Phylogenomics_OMA_AGORA/Run_AGORA_generic.sh

# karyotype / synteny plots
Agora/src/misc.compareGenomes.py <genesA.list.bz2> <genesB.list.bz2> <ancGenes.list.bz2> -mode=drawKaryotype -minChrSize=50 -karyo:landscape=True > <karyo.ps>
python scripts/5_Phylogenomics_OMA_AGORA/Get_alignFile_SyntenyPlotteR.py <in> <align.txt> <chrom.txt> <ref> <target>
Rscript scripts/5_Phylogenomics_OMA_AGORA/Plot_synteny_HOGs.R

---
## 6. Whole-genome alignments & structural-variant identification

**Primary pipeline:** *A. biaculeatus* (`BIA`, GCA_053813585.1) reference,
**minimap2** (v2.28) alignments, **SyRI** (v1.6.3) SV calling, multi-species
merge at **1 kb**. Robustness re-runs with the *A. clarkii* reference and with
**MUMmer4** are summarised in Section 7.7.

### 6.1 Reference-genome whole-genome alignments
Alignments among the four chromosome-level clownfish references (*A. biaculeatus* GCA_053813585.1,
*A. clarkii* GCA_027123335.1, *A. ocellaris* GCA_022539595.1, *A. percula* GCA_003047355.2) with minimap2 and  visualised with
`pafCoordsDotPlotly.R` (dotPlotly). Additional alignments between the *A. biaculeatus* reference and the assembly of *A. biaculeatus* generated in this study (Supplementary Figs S21–S22).

In [ ]:
# minimap2
sbatch scripts/6_WGA_and_SV/Run_Minimap2.sh <workdir> <out_prefix> <reference.fa> <query.fa> <out_prefix>

# plots
Rscript scripts/6_WGA_and_SV/pafCoordsDotPlotly.Plotspdf.R  <paf>  <out_prefix>

### 6.2 Input preparation for SyRI
SyRI requires matching chromosome sets. Assembly scaffolds are named ``scf<chrom>.<part>`` (part 1 = longest scaffold of that chromosome, see Section 2.8). Chromosomes reverse-complemented relative to
the reference are flipped (`metadata/ReverseComplement_Species.txt`,
`metadata/ChromosomesRename_and_RC.txt`); 
Chromosome ↔ accession map: `metadata/CLA_vs_BIARef.mapids.txt`.

In [ ]:
# reverse-complement the flagged chromosomes or scaffolds
python scripts/6_WGA_and_SV/Get_reverse_complement.py <in.fasta> <out.fasta> <ids_to_flip_csv>

# per species: keep only chromosomes / longest scaffolds 
python scripts/6_WGA_and_SV/Extract_longest_chromosomes.py <in_assembly.fasta> <species> <out_dir> <reverse_complement_table.txt>

### 6.3 SyRI vs the *A. biaculeatus* reference

In [ ]:
sbatch scripts/6_WGA_and_SV/Run_syri.sh <workdir> <out_prefix> <reference.fa> <query.fa>

# plots
plotsr --sr <syri.out> --genomes <genomes.txt> -o <out.pdf>

### 6.4 Per-species SV statistics and plotting
Count and cumulative length of each SV class (INV, DUP, TRANS, INS, DEL >50 bp) relative to *A. biaculeatus*.

In [ ]:
# Analyzed genome length
python scripts/6_WGA_and_SV/Get_AnalyzedGenomeLength.py <analysed_genome.fa> <species>

# Statistics from SyRI
python scripts/6_WGA_and_SV/Extract_stats_from_SyRI.py <syri.summary> <species>

# Sum long INDELs
python scripts/6_WGA_and_SV/Sum_long_INDELs.py <syri.out> <species>

# Plotting performed with the script: 
# scripts/6_WGA_and_SV/Final_Plots_For_SV_Statistics.Alltogether.R


### 6.5 VCF reformatting, splitting & multi-species merge (1 kb)
SyRI output reformatted to VCF (`Reformat_SyRI_vcf.py`, adapted from SyRI) and split
into structural rearrangements (`SR`), syntenic (`SYN`), short variants (`ShV`) and
alignment (`AL`). HDR / CPG / CPL / TDM records removed from `ShV`. Files bgzipped +
**tabix** (v1.21) indexed and merged with **`vcf-merge`** (VCFtools). Variants of the
same class with start coordinates **within 1 kb** across species are treated as one.

In [ ]:
# Reformatting and splitting by variant type
python scripts/6_WGA_and_SV/Reformat_SyRI_vcf.py <syri.out> <out.vcf> <workdir> <species> <reference.fa>

python scripts/6_WGA_and_SV/Split_vcf_SyRI.py -i <in.vcf> -o <out_prefix>

# Structural Rearrangements (SR)
# split SR by type + chromosome, merge across species
bash scripts/6_WGA_and_SV/Split_vcf_by_SVtype.sh <species_list.txt> <workdir> <out_dir> <input_pattern> <svtype>

# Merge across species, for each SVtype and chrom
for chrom in chr1 chr2 ... chr24; do
  for SVtype in DUP INVDP TRANS INVTR INV; do
    vcf-merge <"$chrom"."$SVtype".vcf.gz> > <merged."$chrom"."$SVtype".vcf>
  done
done 

# Further merging at 1kb and summary txt files of SVs
python scripts/6_WGA_and_SV/Get_SV_SummaryFile.1kbMerge.py <in.vcf> <chrom> <svtype> <overlap_thresh=1000> <out_prefix>

# Short Variants (ShV)
# clean ShV, compress + index
grep -vE 'HDR|CPG|CPL|TDM' <in.vcf> > <out.vcf>
bgzip -c <vcf> > <vcf.gz> && tabix -p vcf <vcf.gz>

# Merge ShV files by chromosome 
bash   scripts/6_WGA_and_SV/Merge_chromVcf.sh <chrom_list.txt> <shv_by_chrom_dir> <in_prefix> <out_prefix>

# Get SNPs, short INDELS (< 50bp) and larger INDELS (> 50 bp). 
python scripts/6_WGA_and_SV/Filter_ShV_files.py <merged_shv.vcf> <chrom> <out_prefix>

### 6.6 SV summaries & PCoA
SV summary,  **PCoA** (`cmdscale`) on **Jaccard** distances **vegan** v2.7-3). For all SVs and per class.

In [ ]:
# Summmary statistics, PCoA and plotting performed with the script:
# scripts/6_WGA_and_SV/Final_Plots_For_SV_Statistics.Alltogether.R

### 6.7 Robustness - *A. clarkii* reference and MUMmer4
The full pipeline (6.2–6.6) was repeated with (i) the *A. clarkii* reference and
(ii) **MUMmer4** (v4.0.0) in place of minimap2, for both references. Results were
consistent (manuscript: *"data not shown"*). This is not a separate ad hoc check —
it reuses the same working directory and scripts as 7.1–7.6, run a second (and
third) time with different inputs:

* **Reference choice** (*A. biaculeatus* vs *A. clarkii*, both with minimap2):
  Sections 6.2–6.6 were literally re-executed against the *A. clarkii* reference,
  writing to a directory tree parallel to the *A. biaculeatus*. 
  The scripts `Final_Plots_For_SV_Statistics.Alltogether.R` (Section 7.6) also allow to compute statistics and plot results for both references. 

* **Aligner choice** (minimap2 vs MUMmer4, checked for both references): Sections 6.2-6.6 were re-executed for both reference on results obtained with MUMmer. 


In [ ]:
# Aligner-robustness check: MUMmer4 instead of minimap2, both references
sbatch scripts/6_WGA_and_SV/Run_Mummer.sh <workdir> <reference.fa> <query.fa> <out_prefix>
# Then sections 6.2 to 6.6

# Statistics for the 4 methods were obtained with the script 6_/Summarize_Stats_MergedSV.By_Methods.1kb.R 
# and Plot_SV_By_Species_By_Methods.R

---
## 7. SV annotation & interactive catalogue

Each SV's coordinates are intersected with the *A. biaculeatus* gene annotation
(**GenomicRanges** v1.62.1, **rtracklayer** v1.70.1); SVs overlapping ≥1 CDS feature
are flagged and the gene IDs recorded. As no functional annotation is available for
the *A. biaculeatus* reference, its proteome was searched against NCBI **nr** with
**DIAMOND blastp** (v2.0.15), keeping the best hit as a function proxy.

`Final_GetATLAS_SVs.R` builds the annotated SV spreadsheets
(`BIA.{Private,Shared}SV.*`); `generate_sv_database.py` (openpyxl) turns them
into a self-contained interactive HTML table (**DataTables** 1.13.8 + Bootstrap
5.3.2) with filtering by category / type / chromosome / species / length / minimum
species, per-row detail modals and CSV export. The Python generator and HTML
interface were developed with the assistance of Claude (Anthropic).

In [ ]:
# 1. annotated SV tables (BIA reference)
Rscript scripts/7_SV_Catalogue/Final_GetATLAS_SVs.R

# 2. interactive HTML catalogue
python scripts/7_SV_Catalogue/generate_sv_database.py       # -> sv_database.html      (A. biaculeatus reference)

---
## 8. SV & sea-anemone host specialization

Species assigned to four host categories following Gaboriau et al. (2025) —
*Entacmaea* (n=5), *Radianthus* (n=4), *Stichodactyla* (n=2), generalists (n≈6);
see `metadata/Host_Categories.txt`. An SV is **host-specific** when present in all
members of a category and absent from all other species. Significance assessed
against equally sized phylogenetic control groups (three *A. akallopisos*–*A. polymnus*
/ *A. sandaracinos*–*A. sebae* pairs for *Stichodactyla*; one
*A. ocellaris*–*A. percula*–*A. perideraion*–*A. sandaracinos* group for *Radianthus*;
no shared SV among *Entacmaea* specialists). 

Repeated with the *A. clarkii* dataset
(Supplementary Fig S28).

In [ ]:
# Input files were obtained with scripts/6_WGA_and_SV/Get_SV_SummaryFile.1kbMerge.py (section 6.5)

# host-specificity test + control groups + figures (canonical)
# scripts/8_SV_and_Host_Specialization/Final_Plots_For_SV_and_Hosts.Alltogether.R

---
## 9. Long inversions & topological inconsistency


### 9.1 Multi-species SNP matrix from SyRI and chromosome topologies
For each species the SyRI SNPs falling in syntenic / inverted / translocated blocks
are kept. Per-chromosome trees (IQ-TREEsame IQ-TREE settings, whole-chromosome SNPs), then
Robinson–Foulds distances to the HOG species tree (**ape**) and classical MDS
(`cmdscale`). 

In [ ]:
# Keep only syntenic regions
python scripts/9_Long_Inversions_and_Chrom_Trees/Get_Aligned_regions.perSpecies.py <syri.out> <out.bed>

# whole-chromosome SNP FASTA (max 60% missing per site)
python scripts/9_Long_Inversions_and_Chrom_Trees/vcf_to_fasta.py <chrom> <out.fasta> <aligned_regions_dir> <snp.vcf>

# Make trees for each chromosomes
sbatch --array 1-24 scripts/9_Long_Inversions_and_Chrom_Trees/IQtree_byRegions.sh <fasta_file_list.txt> <wd>

# Calculated RF_and_MDS and plot
Rscript scripts/9_Long_Inversions_and_Chrom_Trees/RF_MDS_Heatmap_ChromAndINV.R

### 9.2 Inversion trees & monophyly test
Inversions **>100 kb** present in **≥2 species**; per inversion the SNP subset is
extracted, species with >60 % missing data dropped, regions with ≥9 remaining
species kept. Trees obtained with **IQ-TREE** (v2.2.2.7).

In [ ]:
# SNP FASTA per inversion chunk
python scripts/9_Long_Inversions_and_Chrom_Trees/vcf_to_fasta.py <chrom> <start> <stop> <out.fasta> <aligned_regions_dir> <snp.vcf>

# trees
sbatch scripts/9_Long_Inversions_and_Chrom_Trees/IQtree_byRegions.sh <fasta_file_list.txt> <wd>

# Check monophyletic inversions
# Rscript scripts/9_Long_Inversions_and_Chrom_Trees/check_monophyly.R


### 9.3 Focal inversions — breakpoint split-read genotyping
Two inversions (CM132566.1 / chr18 and CM132553.1 / chr9, manuscript Fig. 3):
breakpoints located from each species' pairwise alignment to *A. biaculeatus*; HiFi reads overlapping a 1-kb window at each breakpoint retrieved (**SAMtools** v1.19.2, see also section 3.3)
and classified as continuous (reference haplotype) vs split (SA tag → inverted
haplotype). Split-read fraction ≈ genotype at the breakpoint. Mapping quality checked
visually with **wally** (v0.6.1).

In [ ]:
# Get breakpoint coordinates per species from SyRI INVAL records
grep "<scaffold>" <syri.out> | grep INVAL | cut -f10 | sort -u

# Retrieve mapped reads from (Section 3.3)
# Check mapping at breakpoints
bash scripts/9_Long_Inversions_and_Chrom_Trees/spanning_reads_breakpoint.sh <breakpoints.txt>  <bam_dir> <bam_suffix> <window_bp> <out_dir>

# same for the second focal inversion

### 9.4 Breakpoint repeat content
For each breakpoint: does it overlap an annotated repeat, and is the 5-kb window
around it enriched/depleted for repeat content vs the chromosome tiled into 5-kb
windows? Two-tailed empirical p-value.

In [ ]:
python scripts/9_Long_Inversions_and_Chrom_Trees/repeat_breakpoint_analysis.py <species> <scaffold> <start> <stop> <outprefix>
python scripts/9_Long_Inversions_and_Chrom_Trees/repeat_5kb_chromosome_tiling_test.py <species> <scaffold> <start> <stop> <outprefix>
python scripts/9_Long_Inversions_and_Chrom_Trees/save_permutation_values_and_plot.py <out>

### 9.5 Genes at breakpoints & within inversions + GO enrichment
Genes spanning the breakpoints and within 5 kb either side; protein-coding genes
inside each inversion → **TopGO** (v2.62.0) GO enrichment per inversion and species
(Fisher exact, `weight01`, min node 5, P<0.05, no multiple-testing correction; terms
enriched in *all* species retained).

In [ ]:
# Get genes
python  scripts/9_Long_Inversions_and_Chrom_Trees/extract_inversion_genes.py <species> <scaffold> <start> <stop> <outprefix>

# Run TopGO for each species 
Rscript scripts/9_Long_Inversions_and_Chrom_Trees/run_topGO_perSpecies.R
Rscript scripts/9_Long_Inversions_and_Chrom_Trees/compare_topGO_results.R